# 05 Feature Engineering and Model-Ready Data

This notebook prepares leakage-aware features for a future return-level simulated abuse detection phase. It does not train or evaluate any model.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Run this notebook from inside the EcomPolice project.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CLEAN_DIR = PROJECT_ROOT / "data" / "synthetic" / "clean"
MESSY_DIR = PROJECT_ROOT / "data" / "synthetic" / "messy"
LABELS_DIR = PROJECT_ROOT / "data" / "synthetic" / "labels"
MANIFESTS_DIR = PROJECT_ROOT / "data" / "synthetic" / "manifests"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def load_tables(base_dir: Path) -> dict[str, pd.DataFrame]:
    return {
        "customers": pd.read_csv(base_dir / "customers.csv"),
        "products": pd.read_csv(base_dir / "products.csv"),
        "orders": pd.read_csv(base_dir / "orders.csv"),
        "deliveries": pd.read_csv(base_dir / "deliveries.csv"),
        "returns": pd.read_csv(base_dir / "returns.csv"),
    }

from synthetic_data.build_features import build_order_level_table, build_pre_return_features
from synthetic_data.preprocessing import build_model_targets
clean = load_tables(CLEAN_DIR)
labels = pd.read_csv(LABELS_DIR / 'order_ground_truth.csv')


## Prediction Point

The main future task is return-level simulated abuse detection. Pre-return features are calculated as of the current order date and use only older customer history. Post-return investigation fields such as return date, return reason, refund amount, and scenario labels are excluded from `X`.


In [2]:
order_level = build_order_level_table(clean['customers'], clean['products'], clean['orders'], clean['deliveries'], clean['returns'])
features = build_pre_return_features(order_level)
targets = build_model_targets(labels)
features['Order_Date'] = pd.to_datetime(features['Order_Date'])
display(features.head())
display(targets.head())


,Order_ID,Customer_ID,Product_ID,Order_Date,Quantity,Unit_Price_INR,Discount_Pct,Order_Value_INR,Payment_Method,Shipping_Method,...,Gender,Loyalty_Tier,Customer_Order_Count_Before,Prior_Return_Count,Historical_Return_Rate,Returns_Previous_30_Days,Returns_Previous_90_Days,Previous_High_Value_Return_Count,Historical_Defective_Claim_Count,Historical_Avg_Return_Duration_Days
0,ORD0000334,CUST001119,PROD00100,2024-01-01,1,1999.00,7.53,1848.48,Cash on Delivery,Standard,...,Female,Silver,0,0,0.0,0,0,0,0,NaN
1,ORD0000975,CUST002132,PROD00327,2024-01-01,1,45820.57,0.00,45820.57,UPI,Standard,...,Female,Gold,0,0,0.0,0,0,0,0,NaN
2,ORD0001915,CUST000116,PROD00333,2024-01-01,1,15529.78,9.94,13986.12,UPI,Standard,...,Female,Bronze,0,0,0.0,0,0,0,0,NaN
3,ORD0002062,CUST001305,PROD00469,2024-01-01,2,2978.42,13.60,5146.71,Debit Card,Standard,...,Male,Bronze,0,0,0.0,0,0,0,0,NaN
4,ORD0002196,CUST000920,PROD00032,2024-01-01,1,74681.85,14.88,63569.19,UPI,Express,...,Female,Silver,0,0,0.0,0,0,0,0,NaN


,Order_ID,Customer_ID,Scenario_Label,Simulated_Abuse_Indicator
0,ORD0000334,CUST001119,REPEATED_DEFECT_CLAIMS,1
1,ORD0000975,CUST002132,MIXED_SUSPICIOUS_PATTERN,1
2,ORD0001915,CUST000116,NORMAL,0
3,ORD0002062,CUST001305,HIGH_VALUE_RETURN_PATTERN,1
4,ORD0002196,CUST000920,NORMAL,0


In [3]:
forbidden = {'Scenario_Label','Simulated_Abuse_Indicator','Return_Date','Return_Reason','Return_Status','Refund_Amount_INR'}
leakage_check = pd.Series({
    'forbidden_columns_in_features': sorted(forbidden.intersection(features.columns)),
    'feature_rows': len(features),
    'target_rows': len(targets),
    'order_id_alignment': set(features['Order_ID']) == set(targets['Order_ID']),
})
display(leakage_check)


forbidden_columns_in_features       []
feature_rows                     20000
target_rows                      20000
order_id_alignment                True
dtype: object

In [4]:
violations = []
for customer_id, group in features.sort_values(['Customer_ID','Order_Date','Order_ID']).groupby('Customer_ID'):
    for _, row in group.iterrows():
        prior = group[group['Order_Date'] < row['Order_Date']]
        if row['Customer_Order_Count_Before'] != len(prior):
            violations.append(row['Order_ID'])
            break
print('Chronological feature violations:', len(violations))


Chronological feature violations: 0


In [5]:
pre_return_feature_columns = [col for col in features.columns if col not in ['Order_ID', 'Customer_ID', 'Product_ID', 'Order_Date']]
data_dictionary = pd.DataFrame({'column': features.columns})
data_dictionary['role'] = np.where(data_dictionary['column'].isin(['Order_ID','Customer_ID','Product_ID']), 'stable identifier', 'pre-return feature')
data_dictionary.loc[data_dictionary['column'].eq('Order_Date'), 'role'] = 'prediction timestamp'
display(data_dictionary)


,column,role
0,Order_ID,stable identifier
1,Customer_ID,stable identifier
2,Product_ID,stable identifier
3,Order_Date,prediction timestamp
4,Quantity,pre-return feature
5,Unit_Price_INR,pre-return feature
6,Discount_Pct,pre-return feature
7,Order_Value_INR,pre-return feature
8,Payment_Method,pre-return feature
9,Shipping_Method,pre-return feature


In [6]:
features.to_csv(PROCESSED_DIR / 'pre_return_features.csv', index=False)
targets.to_csv(PROCESSED_DIR / 'model_targets.csv', index=False)
data_dictionary.to_csv(PROCESSED_DIR / 'feature_data_dictionary.csv', index=False)
report = [
    '# Feature Engineering Report',
    '## Prediction Point',
    'Features are available at the current order date before the current return outcome is known.',
    '## Leakage Controls',
    f'- Forbidden columns in features: {sorted(forbidden.intersection(features.columns))}',
    f'- Feature-target order alignment: {set(features["Order_ID"]) == set(targets["Order_ID"])}',
    f'- Chronological feature violations found: {len(violations)}',
    '## Outputs',
    '- `data/processed/pre_return_features.csv`',
    '- `data/processed/model_targets.csv`',
    '- `data/processed/feature_data_dictionary.csv`',
    '## Limitations',
    '- Labels are generator-defined simulated scenarios, not real verified fraud labels.',
    '- No model training is performed in this phase.',
]
(REPORTS_DIR / 'feature_engineering_report.md').write_text('\n\n'.join(report), encoding='utf-8')
print('Saved feature outputs and report.')


Saved feature outputs and report.


## Summary

The model-ready files keep `X` and `y` separate. Historical customer features use only earlier transactions, and post-return information remains outside the pre-return feature set.
